# SPATIAL INTELLIGENCE - VISIBILITY / ISOVIST ANALYSIS

Compute isovists (viewsheds) across the prison floor plan and derive a visibility graph analysis (VGA) heatmap.

## 1. Import the needed libraries

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy version

In [ ]:
print("This notebook requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

## 3. Configuration

In [ ]:
import os

renderer = "vscode"

# Paths
BASE_DIR = r"E:\IAAC Local GIT Repositories\Graph ML - Environment\Assignment-02_RamonGarcia"
BREP_PATH = os.path.join(BASE_DIR, "prison_clean.brep")
ASSETS_DIR = os.path.join(BASE_DIR, "Assets")
os.makedirs(ASSETS_DIR, exist_ok=True)

GRID_SIZE = 2
ISOVIST_GRID_SIZE = 10

# Image saving - the orchestrator sets this to True
SAVE_IMAGES = True

def save_fig(fig, filename):
    # Save a plotly figure to the assets directory as PNG.
    if not SAVE_IMAGES or fig is None:
        return
    try:
        path = os.path.join(ASSETS_DIR, filename)
        fig.write_image(path, width=1600, height=1200, scale=2)
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Could not save {filename}: {e}")

## 4. Utility functions

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)

## 5. Load the floor plan and create dual grids

We use two grids: a **coarse vertex grid** for isovist viewpoints and a **fine edge grid** for the analysis shell.

In [ ]:
# Load the cleaned floor plan
floor_plan = Topology.ByBREPPath(BREP_PATH)

# Bounding rectangle
b_r = Wire.BoundingRectangle(floor_plan)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")

# Coarse grid for isovist viewpoints
uRange1 = list(range(0, int(width) + ISOVIST_GRID_SIZE, ISOVIST_GRID_SIZE))
vRange1 = list(range(0, int(length) + ISOVIST_GRID_SIZE, ISOVIST_GRID_SIZE))
grid_viewpoints = Grid.VerticesByDistances(floor_plan, clip=True, uRange=uRange1, vRange=vRange1)

# Fine grid for analysis
uRange2 = list(range(0, int(width) + GRID_SIZE, GRID_SIZE))
vRange2 = list(range(0, int(length) + GRID_SIZE, GRID_SIZE))
grid_edges = Grid.EdgesByDistances(floor_plan, clip=True, uRange=uRange2, vRange=vRange2)

## 6. Slice the floor plan and derive the analysis graph

In [ ]:
shell = Topology.Slice(floor_plan, grid_edges)
faces = Topology.Faces(shell)
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_" + str(i + 1))
    f = Topology.SetDictionary(f, d)
print(f"Shell: {len(faces)} faces")

analysis_graph = Graph.ByTopology(shell)
g_verts = Graph.Vertices(analysis_graph)
iso_verts = Topology.Vertices(grid_viewpoints)
print(f"Isovist viewpoints: {len(iso_verts)}")

## 7. Compute isovists at each viewpoint

An isovist is the area visible from a given point within the floor plan. This step is computationally intensive.

In [ ]:
isovists = []
for i, v in enumerate(iso_verts):
    isovist = Face.Isovist(floor_plan, v)
    isovists.append(isovist)
    if (i + 1) % 10 == 0:
        print(f"  Computed {i + 1}/{len(iso_verts)} isovists...")
print(f"Done. {sum(1 for iso in isovists if iso is not None)}/{len(isovists)} valid isovists")

## 8. Show isovists overlaid on the floor plan

In [ ]:
valid_isovists = [iso for iso in isovists if iso is not None]
fig = Topology.Show(floor_plan, *valid_isovists,
              faceOpacity=0.6,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)
save_fig(fig, "13_isovists_overlay.png")

## 9. Compute visibility scores

For each isovist, count how many grid vertices fall inside it. This gives a measure of how much of the space is visible from each viewpoint.

In [ ]:
new_verts = []
n_list = []
for i, iso in enumerate(isovists):
    if iso is None:
        continue
    v = iso_verts[i]
    # Handle both Face and Cluster results from Face.Isovist
    if Topology.IsInstance(iso, "Cluster"):
        iso_faces = Cluster.Faces(iso)
        iso = iso_faces[0] if iso_faces else None
        if iso is None:
            continue
    b_list = Vertex.IsInternal2D(g_verts, iso)
    b_list = [b for b in b_list if b]
    n = len(b_list)
    n_list.append(n)
    d = Dictionary.ByKeyValue("visibility", n)
    v = Topology.SetDictionary(v, d)
    new_verts.append(v)

print(f"Visibility range: {min(n_list)} - {max(n_list)}")
print(f"Mean visibility: {sum(n_list) / len(n_list):.1f}")

## 10. Interpolate visibility to the full grid and color-map

In [ ]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    v = Topology.SetDictionary(v, d)

Transfer to the shell faces

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

## 11. Show visibility heatmap

In [ ]:
fig = Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0, 0, 6],
              backgroundColor="black",
              width=800, height=600,
              renderer=renderer)
save_fig(fig, "14_visibility_heatmap.png")